# NexCart Intelligence — Collaborative Filtering (SVD)
**Dataset:** Amazon Fine Food Reviews (568K reviews)
**Model:** SVD via Surprise library
**Target:** RMSE < 1.0 | Precision@10 > 0.25

In [1]:
import pandas as pd              # Data manipulation and analysis (tables, CSVs, etc.)
import numpy as np               # Numerical computing (arrays, math operations)
import pickle                    # Save/load Python objects (e.g., trained models)
import os                        # File and directory operations
from surprise import SVD, Dataset, Reader   # Surprise library: SVD algorithm, dataset handling, and reader for ratings
from surprise.model_selection import cross_validate, train_test_split  
# Tools for splitting data into train/test sets and running cross-validation
from surprise import accuracy     # Functions to measure model performance (RMSE, MAE, etc.)
from collections import defaultdict  # Default dictionary for convenient data structures


## Step 1 — Load & Sample Dataset

In [2]:
df = pd.read_csv('../data/Reviews.csv')
print(f"Full dataset shape: {df.shape}")
print(f"File loaded correctly: {'✅' if len(df) > 500_000 else '❌ Wrong file'}")
print(df.head(3))

Full dataset shape: (568454, 10)
File loaded correctly: ✅
   Id   ProductId          UserId                      ProfileName  \
0   1  B001E4KFG0  A3SGXH7AUHU8GW                       delmartian   
1   2  B00813GRG4  A1D87F6ZCVE5NK                           dll pa   
2   3  B000LQOCH0   ABXLMWJIXXAIN  Natalia Corres "Natalia Corres"   

   HelpfulnessNumerator  HelpfulnessDenominator  Score        Time  \
0                     1                       1      5  1303862400   
1                     0                       0      1  1346976000   
2                     1                       1      4  1219017600   

                 Summary                                               Text  
0  Good Quality Dog Food  I have bought several of the Vitality canned d...  
1      Not as Advertised  Product arrived labeled as Jumbo Salted Peanut...  
2  "Delight" says it all  This is a confection that has been around a fe...  


In [3]:
# Keep only needed columns
df = df[['UserId', 'ProductId', 'Score']].dropna()

# Remove duplicates — same user rating same product twice
df = df.drop_duplicates(subset=['UserId', 'ProductId'])

# Sample max available or 300K whichever is smaller
sample_size = min(300_000, len(df))
df = df.sample(n=sample_size, random_state=42)

print(f"Available rows after dedup: {len(df)}")
print(f"Sampling: {sample_size} rows")

print(f"Sampled shape: {df.shape}")
print(f"Unique users: {df.UserId.nunique()}")
print(f"Unique products: {df.ProductId.nunique()}")
print(f"Score distribution:\n{df.Score.value_counts().sort_index()}")

Available rows after dedup: 300000
Sampling: 300000 rows
Sampled shape: (300000, 3)
Unique users: 165614
Unique products: 56540
Score distribution:
Score
1     27507
2     15847
3     22586
4     42506
5    191554
Name: count, dtype: int64


## Step 2 — Apply Min Interaction Filters

In [4]:
# Users with < 5 reviews → cold start territory
user_counts = df['UserId'].value_counts()
active_users = user_counts[user_counts >= 5].index
df = df[df['UserId'].isin(active_users)]

# Products with < 10 reviews → too sparse for SVD
product_counts = df['ProductId'].value_counts()
active_products = product_counts[product_counts >= 10].index
df = df[df['ProductId'].isin(active_products)]

print(f"After filters shape: {df.shape}")
print(f"Active users: {df.UserId.nunique()}")
print(f"Active products: {df.ProductId.nunique()}")

After filters shape: (53252, 3)
Active users: 8456
Active products: 1347


## Step 3 — Build Surprise Dataset

In [6]:
reader = Reader(rating_scale=(1, 5))

data = Dataset.load_from_df(
    df[['UserId', 'ProductId', 'Score']],
    reader
)

print("Surprise dataset built ✅")

Surprise dataset built ✅


## Step 4 — 5-Fold Cross Validation

In [7]:
svd = SVD(n_factors=150, n_epochs=30, lr_all=0.005, reg_all=0.02, biased=True, random_state=42)

cv_results = cross_validate(
    svd,
    data,
    measures=['RMSE', 'MAE'],
    cv=5,
    verbose=True
)

print(f"\nMean RMSE: {cv_results['test_rmse'].mean():.4f}")
print(f"Mean MAE:  {cv_results['test_mae'].mean():.4f}")
print(f"\nTarget RMSE < 1.0: {'✅ PASSED' if cv_results['test_rmse'].mean() < 1.0 else '❌ FAILED'}")

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.7860  0.7746  0.7807  0.7974  0.7872  0.7852  0.0076  
MAE (testset)     0.5661  0.5626  0.5690  0.5768  0.5704  0.5690  0.0047  
Fit time          1.08    1.26    1.21    1.30    1.10    1.19    0.09    
Test time         0.06    0.10    0.06    0.12    0.06    0.08    0.03    

Mean RMSE: 0.7852
Mean MAE:  0.5690

Target RMSE < 1.0: ✅ PASSED


## Step 5 — Train/Test Split + Precision@10

In [8]:
trainset, testset = train_test_split(data, test_size=0.20, random_state=42)

svd = SVD(n_factors=150, n_epochs=30, lr_all=0.005, reg_all=0.02, biased=True, random_state=42)
svd.fit(trainset)

predictions = svd.test(testset)
print(f"RMSE on test set: {accuracy.rmse(predictions):.4f}")
print(f"MAE  on test set: {accuracy.mae(predictions):.4f}")

RMSE: 0.7753
RMSE on test set: 0.7753
MAE:  0.5588
MAE  on test set: 0.5588


In [10]:
def precision_at_k(predictions, k=5, threshold=3.5):
    """
    Precision@K — only evaluate users who have
    at least k items in test set for fair evaluation
    """
    user_est_true = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions = []
    for uid, user_ratings in user_est_true.items():
        # Skip users with fewer than k test items
        if len(user_ratings) < k:
            continue

        # Sort by estimated rating descending
        user_ratings.sort(key=lambda x: x[0], reverse=True)

        # Top-K
        top_k = user_ratings[:k]

        # How many in top-K were actually positive?
        n_relevant = sum(1 for _, true_r in top_k if true_r >= threshold)
        precisions.append(n_relevant / k)

    print(f"Users evaluated: {len(precisions)}")
    return sum(precisions) / len(precisions) if precisions else 0.0


p_at_5 = precision_at_k(predictions, k=5, threshold=3.5)
print(f"Precision@5:  {p_at_5:.4f}")
print(f"Target > 0.25: {'✅ PASSED' if p_at_5 > 0.25 else '❌ FAILED'}")

Users evaluated: 212
Precision@5:  0.7934
Target > 0.25: ✅ PASSED


## Step 6 — Train Final Model on Full Dataset

In [11]:
full_trainset = data.build_full_trainset()
svd_final = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
svd_final.fit(full_trainset)

print("Final SVD model trained on full dataset ✅")

Final SVD model trained on full dataset ✅


## Step 7 — Save Model + Metadata

In [12]:
os.makedirs('../saved_models', exist_ok=True)

with open('../saved_models/svd_model.pkl', 'wb') as f:
    pickle.dump(svd_final, f)

product_meta = df.groupby('ProductId').agg(
    review_count=('Score', 'count'),
    avg_rating=('Score', 'mean')
).reset_index()

with open('../saved_models/product_meta.pkl', 'wb') as f:
    pickle.dump(product_meta, f)

known_users = set(df['UserId'].unique())
with open('../saved_models/known_users.pkl', 'wb') as f:
    pickle.dump(known_users, f)

known_products = list(df['ProductId'].unique())
with open('../saved_models/known_products.pkl', 'wb') as f:
    pickle.dump(known_products, f)

print("svd_model.pkl ✅")
print("product_meta.pkl ✅")
print("known_users.pkl ✅")
print("known_products.pkl ✅")

svd_model.pkl ✅
product_meta.pkl ✅
known_users.pkl ✅
known_products.pkl ✅


## Step 8 — Quick Inference Test

In [13]:
sample_user = df['UserId'].iloc[0]
sample_product = df['ProductId'].iloc[0]

pred = svd_final.predict(sample_user, sample_product)
print(f"User:             {pred.uid}")
print(f"Product:          {pred.iid}")
print(f"Predicted rating: {pred.est:.2f}")
print(f"\nInference working ✅")    

User:             A27X2M44NJTI1I
Product:          B003YK8YL0
Predicted rating: 3.69

Inference working ✅
